In [16]:
#Importing data
import pandas as pd

train_df = pd.read_csv(
    "train_FD001.txt",
    sep=r"\s+",
    header=None
)

test_df = pd.read_csv(
    "test_FD001.txt",
    sep=r"\s+",
    header=None
)

In [17]:
#Adding column names to data

column_names = (
    ["unit_number", "time_in_cycles"] +
    [f"operational_setting_{i}" for i in range(1, 4)] +
    [f"sensor_measurement_{i}" for i in range(1, 22)]
)

train_df.columns = column_names
test_df.columns = column_names

In [18]:
#Checking for missing values
print(train_df.isna().sum().sum())
print(test_df.isna().sum().sum())

0
0


In [22]:
#Removed sensors with 0 variance
sensor_cols = [
    col for col in train_df.columns
    if "sensor_measurement" in col
]

for col in sensor_cols:
    print(col, train_df[col].var())

constant_sensors = [
    col for col in sensor_cols
    if train_df[col].nunique() == 1
]
train_df = train_df.drop(columns=constant_sensors)
test_df = test_df.drop(columns=constant_sensors)

sensor_measurement_2 0.022685918937648035
sensor_measurement_3 0.0178659590588091
sensor_measurement_4 0.023084117940328313
sensor_measurement_6 0.019292790877293044
sensor_measurement_7 0.020313926801759336
sensor_measurement_8 0.01156781040719895
sensor_measurement_9 0.009818545429260582
sensor_measurement_11 0.02527482940480227
sensor_measurement_12 0.024730975325628863
sensor_measurement_13 0.011185835716392424
sensor_measurement_14 0.009690913945149714
sensor_measurement_15 0.02082412008552804
sensor_measurement_17 0.016657409072308556
sensor_measurement_20 0.01963179567879846
sensor_measurement_21 0.022343221346280584


In [23]:
max_cycles = (
    train_df.groupby("unit_number")
            ["time_in_cycles"]
            .transform("max")
)

train_df["RUL"] = (
    max_cycles -
    train_df["time_in_cycles"]
)

In [24]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

feature_columns = [
    col for col in train_df.columns
    if col not in ["unit_number", "RUL"]
]

train_df[feature_columns] = scaler.fit_transform(
    train_df[feature_columns]
)

test_df[feature_columns] = scaler.transform(
    test_df[feature_columns]
)

print(test_df)





       unit_number  time_in_cycles  operational_setting_1  \
0                1        0.000000               0.632184   
1                1        0.002770               0.344828   
2                1        0.005540               0.517241   
3                1        0.008310               0.741379   
4                1        0.011080               0.580460   
...            ...             ...                    ...   
13091          100        0.534626               0.781609   
13092          100        0.537396               0.436782   
13093          100        0.540166               0.465517   
13094          100        0.542936               0.281609   
13095          100        0.545706               0.574713   

       operational_setting_2  operational_setting_3  sensor_measurement_2  \
0                   0.750000                    0.0              0.545181   
1                   0.250000                    0.0              0.150602   
2                   0.583333        